In [1]:
# ============================================================
# 1.라이브러리 불러오기 
# ============================================================

import os
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================

train = pd.read_csv("../../data/interim/train_missing_handled.csv")

In [3]:
# ============================================================
# 3. 4번 EDA에서 선택한 구간 기준을 검정용 임시 변수로 생성
# ============================================================

test_df = train.copy()

test_df['AgeBand'] = pd.cut(
    test_df['Age'],
    bins=[0,10,20,40,60,80],
    labels=['Child','Teen','YounAdult','Adult','Senior'],
    include_lowest=True
)

fare_bins = pd.qcut(
    test_df['Fare'],
    q=4,
    retbins=True,
    duplicates='drop'
)[1]

fare_labels_all = ['Low', 'MidLow', 'MidHigh', 'High']
fare_labels = fare_labels_all[:len(fare_bins)-1]

test_df['FareBand'] = pd.cut(
    test_df['Fare'],
    bins=fare_bins,
    labels=fare_labels,
    include_lowest=True
)

In [4]:
# ============================================================
# 4. 검정 대상 변수 설정
# ============================================================

cat_cols_for_test = [
    'Sex',
    'Embarked',
    'Pclass',
    'SibSp',
    'Parch',
    'AgeBand',
    'FareBand'
]

num_cols_for_test = [
    'Age',
    'Fare'
]

In [5]:
# ============================================================
# 5. 카이제곱 검정 함수
# ============================================================

def cramers_v(chi2, n, r, k):
    denominator = n * (min(r - 1, k - 1))
    if denominator == 0:
        return np.nan
    return np.sqrt(chi2 / denominator)

def chi_square_test(df, col, target='Survived'):
    contingency_table = pd.crosstab(df[col], df[target])

    chi2, p, dof, expected = chi2_contingency(contingency_table)

    n = contingency_table.sum().sum()
    r, k = contingency_table.shape

    effect_size = cramers_v(chi2, n, r, k)
    low_expected_count = (expected < 5).sum()

    return{
        'variable' : col,
        'test' : 'chi_square',
        'chi2_stat' : chi2,
        'p_value' : p,
        'dof' : dof,
        'effect_size_CramersV' : effect_size,
        'low_expected_cells' : low_expected_count,
        'significant_0.05' : p <= 0.05
    }

In [6]:
# ============================================================
# 6. 범주형 변수 카이제곱 검정 실행
# ============================================================

cat_results = []

for col in cat_cols_for_test:
    cat_results.append(chi_square_test(test_df, col))

cat_test_df = pd.DataFrame(cat_results).sort_values(by='p_value')

display(cat_test_df)

,variable,test,chi2_stat,p_value,dof,effect_size_CramersV,low_expected_cells,significant_0.05
0,Sex,chi_square,260.7170,0.0000,1,0.5409,0,True
2,Pclass,chi_square,102.8890,0.0000,2,0.3398,0,True
6,FareBand,chi_square,80.1739,0.0000,3,0.3000,0,True
3,SibSp,chi_square,37.2718,0.0000,6,0.2045,4,True
1,Embarked,chi_square,25.9645,0.0000,2,0.1707,0,True
4,Parch,chi_square,27.9258,0.0001,6,0.1770,8,True
5,AgeBand,chi_square,15.0937,0.0045,4,0.1302,0,True


In [7]:
# ============================================================
# 7. Welch t-test 함수
# ============================================================

def t_test_by_target(df, col, target='Survived'):
    group1 = df[df[target] == 1][col].dropna()
    group0 = df[df[target] == 0][col].dropna()

    stat, p = ttest_ind(group1, group0, equal_var=False)

    return{
        'variable' : col,
        'test' : 'welch_t_test',
        't_stat' : stat,
        'p_value' : p,
        'mean_survived_1' : group1.mean(),
        'mean_survived_0' : group0.mean(),
        'mean_diff_1_minus_0' : group1.mean() - group0.mean(),
        'significant_0.05' : p <= 0.05
    }

In [8]:
# ============================================================
# 8. 숫자형 변수 t-test 실행
# ============================================================

num_results = []

for col in num_cols_for_test:
    num_results.append(t_test_by_target(test_df, col))

num_test_df = pd.DataFrame(num_results).sort_values(by='p_value')

display(num_test_df)

,variable,test,t_stat,p_value,mean_survived_1,mean_survived_0,mean_diff_1_minus_0,significant_0.05
1,Fare,welch_t_test,6.8391,0.0000,48.3954,22.1179,26.2775,True
0,Age,welch_t_test,-1.6921,0.0911,28.4756,30.0282,-1.5526,False


Fare는 생존 여부에 통계적으로 유의한 차이를 보였으며 생존자가 더 높은 요금을 지불했다. 

반면 Age는 평균 차이가 존재하지만 통계적으로 유의하지 않아 단독 변수로는 영향력이 약한 것으로 판단된다.

In [9]:
# ============================================================
# 9. 결과 저장
# ============================================================

cat_test_df.to_csv("../../output/tables/categorical_statistical_tests.csv", index=False)

num_test_df.to_csv("../../output/tables/numerical_statistical_tests.csv", index=False)

print("통계 검정 결과 저장 완료")

통계 검정 결과 저장 완료
